[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/heat_cycles/heat_cycles.ipynb)

# The Carnot Cycle

This example computes the heat and the work of a Carnot engine: a gas in a cylinder takes in heat from a hot reservoir while it expands, expands further insulated until it has cooled to the cold reservoir's temperature, gives heat to the cold reservoir while it is compressed, and is compressed insulated back to where it started. Its energy returns to where it was, so the work it has done is the heat it took in less the heat it gave up.

A state of the gas is modelled as an antivector of the plane spanned by its entropy and its volume, and the gas as its energy, a function of the state. The energy's derivative is a reading of changes of state, a map `Scalar <- State`; the heat a change takes in is the temperature times the entropy's reading, `temperature * entropy`, and the work it does the pressure times the volume's, `pressure * volume`. Their difference is the energy's derivative, while neither alone is the derivative of anything, which is why heat and work depend on the path taken. Antisymmetrised, the derivatives of the two readings are forms on pairs of changes, which read oriented areas, and they are the same form: around a closed cycle the area enclosed on the temperature–entropy chart, the net heat, equals the area enclosed on the pressure–volume chart, the net work.

In the usual notation the heat and the work are the inexact differentials $\delta Q = T\,dS$ and $\delta W = p\,dV$, with $dU = \delta Q - \delta W$; the equal circulations are $dT \wedge dS = dp \wedge dV$, the identity behind Maxwell's relations, and the engine's efficiency is $W/Q_\text{hot} = 1 - T_\text{cold}/T_\text{hot}$.

Entropy is in $nR$, volume and temperature in reference values, energy in $nRT_0$; work is positive when the gas does it.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples,jax] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import jax
import numpy as np

from numga import Algebra, concatenate
from numga.backend.jax import JaxContext, derivative
from examples.util.animation import show_animation
from examples.thermodynamics.heat_cycles import render

jax.config.update("jax_enable_x64", True)
ga = Algebra("s0v0")
mv = JaxContext(ga, np.float64).multivector
Scalar = ga.gatype.scalar()
State = ga.gatype.antivector()
Reading = ga.gatype((Scalar, State))
AreaReading = ga.gatype((Scalar, State, State))

# The two readings a state is described by.
entropy: Reading = mv.s & State                            # Scalar <- State
volume: Reading = mv.v & State                             # Scalar <- State
# The gas's heat capacity at fixed volume, in nR.
capacity = 1.5

## The cycle

The cycle has four corners in entropy and temperature, and each stroke runs straight between two of them: at the reservoirs' temperatures the gas's entropy changes, in between its temperature changes while its entropy holds. A state at an entropy and a temperature has the volume at which the energy's slope along the entropy is that temperature. The charts show the cycle among the gas's isotherms and dashed adiabats on the left, and among its lines of equal volume and dashed lines of equal pressure on the right.

In [ ]:
def energy(state: State) -> Scalar:
    """The energy of an ideal gas with a constant heat capacity at fixed volume."""
    return capacity * ((entropy(state) - volume(state).log()) / capacity).exp()

temperature_of = lambda states: derivative(energy)(states)(mv.s.dual())
pressure_of = lambda states: -derivative(energy)(states)(mv.v.dual())

def lifted(entropies: Scalar, temperatures: Scalar) -> State:
    """The gas's states at entropies and temperatures."""
    return mv.s.dual() * entropies + mv.v.dual() * (entropies - capacity * temperatures.log()).exp()

def strokes(fraction: Scalar) -> State:
    """The states a fraction of the way along each stroke `[4, ...]`, straight in entropy and temperature."""
    entropies = corner_entropy[:-1] + (corner_entropy[1:] - corner_entropy[:-1]) * fraction
    temperatures = corner_temperature[:-1] + (corner_temperature[1:] - corner_temperature[:-1]) * fraction
    return lifted(entropies, temperatures)

# The corners, the first again at the end: heated at the hot temperature, expanded insulated, cooled at the cold
# temperature, compressed insulated; and the steps along each stroke.
hot, cold, entropy_span = 1.5, 1.0, 1.0
corner_entropy = mv.scalar([[0], [entropy_span], [entropy_span], [0], [0]])[:, None]                  # [5, 1] Scalar
corner_temperature = mv.scalar([[hot], [hot], [cold], [cold], [hot]])[:, None]                        # [5, 1] Scalar
n_steps = 16
ends = mv.scalar(np.broadcast_to(np.arange(1, n_steps + 1) / n_steps, (4, n_steps))[..., None])      # [4, steps] Scalar
cycle = concatenate([strokes(mv.scalar([[0]]))[0, :1], strokes(ends).reshape(-1)])                 # [4 * steps + 1] State
render.draw_cycle(temperature_of, pressure_of, cycle)

## Heat and work along the strokes

The heat a change of state takes in is the reading `temperature * entropy`, the work it does `pressure * volume`, both built from the energy's derivative at the state. Each stroke's tangent is the derivative of the stroke with respect to the fraction along it, bound to a whole stroke, `derivative(strokes)(middles)(mv.scalar([1]))`; the two readings of the tangents at the middles of the steps, summed, give each stroke's heat and work.

In [ ]:
def exchanges(states: State, tangents: State) -> tuple[Scalar, Scalar]:
    """The heat a change of state along each tangent takes in and the work it does."""
    differential = derivative(energy)(states)                                      # [...] Scalar <- State
    heat = differential(mv.s.dual()) * entropy                                     # [...] Scalar <- State
    work = -differential(mv.v.dual()) * volume                                     # [...] Scalar <- State
    return heat(tangents), work(tangents)

middles = mv.scalar(np.broadcast_to((np.arange(n_steps) + 0.5) / n_steps, (4, n_steps))[..., None])  # [4, steps] Scalar
tangents = derivative(strokes)(middles)(mv.scalar([1]))                                             # [4, steps] State
heat_rate, work_rate = exchanges(strokes(middles), tangents)                                        # [4, steps] Scalar
stroke_heat, stroke_work = heat_rate.sum(axis=1) / n_steps, work_rate.sum(axis=1) / n_steps         # [4] Scalar

In [ ]:
print("heat taken in by stroke:", stroke_heat.to_array())
print("work done by stroke:    ", stroke_work.to_array())
print("efficiency:", (stroke_work.sum() / stroke_heat[0]).to_array())

## Equal areas

The heat circulated around a small area of states is the temperature's change wedged with the entropy's, a form on the pairs of changes that span the area, and the work circulated the pressure's change wedged with the volume's. The two forms are equal at every state, so the net heat and the net work around any closed cycle are equal too: the areas marked W on the two charts.

In [ ]:
def circulations(state: State) -> tuple[AreaReading, AreaReading]:
    """The heat and the work circulated around small oriented areas of states."""
    temperature_change = derivative(temperature_of)(state)                         # [...] Scalar <- State
    pressure_change = derivative(pressure_of)(state)                               # [...] Scalar <- State
    heat_area = temperature_change * entropy - entropy * temperature_change        # [...] Scalar <- (State, State)
    work_area = pressure_change * volume - volume * pressure_change                # [...] Scalar <- (State, State)
    return heat_area, work_area

heat_area, work_area = circulations(cycle)                                         # [steps] Scalar <- (State, State)

In [ ]:
print("heat circulated around the unit area, at the corners:", heat_area(mv.s.dual(), mv.v.dual())[::n_steps].to_array())
print("work circulated around the unit area, at the corners:", work_area(mv.s.dual(), mv.v.dual())[::n_steps].to_array())

## Running the engine

The gas runs around the cycle step by step, the piston rising while it is heated and while it expands insulated, falling while it is cooled and compressed.

In [ ]:
def running():
    """The gas's state at each step around the cycle."""
    for index in range(cycle.shape[0]):
        yield cycle[index]

show_animation(render.animate_cycle(temperature_of, pressure_of, cycle, running()), "thermodynamics_heat_cycle",
               duration_ms=70)

Orange marks the heat taken in from the hot reservoir, blue the heat given to the cold one, and grey the insulated strokes. The gas takes in 1.5 units of heat, gives up 1, and does 0.5 units of work, an efficiency of one third, as $1 - T_\text{cold}/T_\text{hot}$ says.

In [ ]:
# checks
np.testing.assert_allclose((cycle[-1] - cycle[0]).kernel, 0, atol=1e-10)
np.testing.assert_allclose(stroke_heat.to_array(), [hot * entropy_span, 0, -cold * entropy_span, 0], atol=1e-10)
np.testing.assert_allclose(stroke_work.sum().to_array(), (hot - cold) * entropy_span, atol=1e-10)
# The energy's change along each stroke is its heat less its work; around the cycle it is none.
energy_change = energy(cycle[n_steps::n_steps]) - energy(cycle[:-1:n_steps])
np.testing.assert_allclose((stroke_heat - stroke_work - energy_change).kernel, 0, atol=1e-10)
np.testing.assert_allclose((heat_area - work_area).kernel, 0, atol=1e-10)